To augment the training data, I implemented a back-translation pipeline using the Hugging Face transformers library. I used pre-trained Transformer-based machine translation models from the Helsinki-NLP/OPUS-MT project (e.g., opus-mt-en-fr and opus-mt-fr-en). By translating the enriched English training instances into French and back into English, I generated semantically similar variations to increase the size of the enriched dataset while maintaining the original label associations.


In [ ]:
import os
os.environ["USE_TF"] = "0"

from transformers import pipeline
import pandas as pd
import os

data_path = "../data/"
input_filename = "train_emoji_enrichi.csv"
output_filename = "train_emoji_enrichi_aug.csv"

df = pd.read_csv(os.path.join(data_path, input_filename))
df.head()


In [ ]:
#afficher la repartition des classes avant augmentation
print(df['emoji'].value_counts().sort_index())
print(f"Total samples: {len(df)}")

In [ ]:
#Initialisation des modeles de traduction
#anglais -> francais puis francais -> anglais
#Helsinki-NLP est leger et adapté pour cette augmentation simple
#Au premier lancement, transformers telecharge les modeles puis les garde en cache.
import os
os.environ["USE_TF"] = "0"

from transformers import pipeline

to_fr = pipeline("translation_en_to_fr", model="Helsinki-NLP/opus-mt-en-fr", framework="pt")
to_en = pipeline("translation_fr_to_en", model="Helsinki-NLP/opus-mt-fr-en", framework="pt")

def back_translate(text):
    print(f"Original: {text}")

    fr = to_fr(text)[0]["translation_text"]
    print(f"Translated to French: {fr}")

    en = to_en(fr)[0]["translation_text"]
    print(f"Back-translated: {en}")

    return en



In [ ]:
#data augmentation seulement sur les classes minoritaires
#0: amour, 1: sport, 4: nourriture
minority_classes = [0, 1, 4]
augmented_data = []

#certaines back-translations changent le sens ou produisent une phrase incoherente
#on les filtre manuellement pour ne pas ajouter de bruit dans le dataset
invalid_augmented_sentences = {
    "The little chicago have won again",
    "let the brunch one day",
    "leaves the exercise",
    "Let's give food together",
    "The first man on the base had the bullet",
    "Do you have a bullet?",
    "You're downstairs for baseball this afternoon",
    "Who's downstairs for a restaurant",
    "He had to run a home race",
}

#garder les phrases deja presentes pour eviter les duplicats exacts
existing_sentences = set(df['sentence'].str.lower().str.strip())

for index, row in df.iterrows():
    if row['emoji'] in minority_classes:
        new_text = back_translate(row['sentence']).strip().rstrip(".")
        new_text_lower = new_text.lower().strip()

        #garder seulement les paraphrases differentes, non deja presentes et coherentes
        if new_text not in invalid_augmented_sentences and new_text_lower not in existing_sentences:
            augmented_data.append({'sentence': new_text, 'emoji': row['emoji']})
            existing_sentences.add(new_text_lower)

len(augmented_data)


In [ ]:
#combiner les donnees enrichies avec les nouvelles phrases
#pas de shuffle pour ajouter les nouvelles instances a la fin du csv original
df_augmented = pd.concat([df, pd.DataFrame(augmented_data)], ignore_index=True)

output_path = os.path.join(data_path, output_filename)
df_augmented.to_csv(output_path, index=False)

print(f"Original enriched dataset size: {len(df)}")
print(f"New generated instances: {len(augmented_data)}")
print(f"Augmented enriched dataset size: {len(df_augmented)}")
print(f"Saved augmented enriched dataset to: {output_path}")


In [ ]:
#afficher la repartition des classes apres augmentation
df_augmented['emoji'].value_counts().sort_index()


In [ ]:
#verifier les nouvelles instances ajoutees a la fin
df_augmented.tail(len(augmented_data))
